# Issue #13: Popularity Feature Engineering

This notebook creates features for predicting movie popularity.

## Step 1: Verify the Input Files

Before loading or joining data, confirm the project location and whether
the required base dataset, `data/processed/movies_metadata.csv`, exists.


In [6]:
from pathlib import Path

# Search the current folder and its parents for the project root.
current_folder = Path.cwd()

project_root = next(
    (
        folder
        for folder in [current_folder, *current_folder.parents]
        if (folder / "data" / "processed").is_dir()
        and (folder / "requirements.txt").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the DSML project folder.")

processed_folder = project_root / "data" / "processed"
base_path = processed_folder / "movies_metadata.csv"

print("Project folder:", project_root)
print("Required base dataset exists:", base_path.is_file())

print("\nAvailable processed CSV files:")
for file_path in sorted(processed_folder.glob("*.csv")):
    print("-", file_path.name)


Project folder: /Users/jasper/css-dsml-fall2027
Required base dataset exists: True

Available processed CSV files:
- credits.csv
- keywords_clean.csv
- keywords_parsed.csv
- movie_text.csv
- movies_metadata.csv
- nested_metadata.csv


## Step 2: Load and Inspect the Base Dataset

**Current: Requirement 1 — Use `data/processed/movies_metadata.csv` as the base dataset.**

(Also checking Requirement 6: Confirm one row per `id`.)

Load the processed metadata and inspect its dimensions, sample rows,
column types, missing IDs, and duplicate IDs.

Source: `data/processed/movies_metadata.csv`.


In [7]:
import pandas as pd

# Load the required base dataset.
movies = pd.read_csv(base_path)

print("Number of rows:", movies.shape[0])
print("Number of columns:", movies.shape[1])

print("\nFirst five movies:")
display(movies.head())

print("\nColumn data types:")
display(movies.dtypes.to_frame(name="data_type"))

print("\nMissing movie IDs:", movies["id"].isna().sum())
print("Duplicate movie IDs:", movies["id"].duplicated().sum())

Number of rows: 45449
Number of columns: 6

First five movies:


,id,release_date,budget,revenue,runtime,popularity
0,862.0,1995-10-30,30000000,373554033.0,81.0,21.946943
1,8844.0,1995-12-15,65000000,262797249.0,104.0,17.015539
2,15602.0,1995-12-22,0,0.0,101.0,11.712900
3,31357.0,1995-12-22,16000000,81452156.0,127.0,3.859495
4,11862.0,1995-02-10,0,76578911.0,106.0,8.387519



Column data types:


,data_type
id,float64
release_date,object
budget,object
revenue,float64
runtime,float64
popularity,float64



Missing movie IDs: 3
Duplicate movie IDs: 15


## Step 3: Inspect Popularity Values

**Current: Requirement 3 — Inspect missing, negative, and suspicious popularity values.**

(Also addressing Requirement 7: Document all feature sources and decisions.)

Check popularity for missing values, nonnumeric entries, infinity,
negative values, zeros, and unusually large values.

Review summary statistics and the ten highest values before deciding
whether any records need special handling. A large value alone does
not prove that a record is incorrect.

This step inspects the data without changing or removing any records.


In [8]:
import numpy as np

# Convert to numbers for inspection without changing the original column.
popularity_numeric = pd.to_numeric(
    movies["popularity"], errors="coerce"
)

missing = movies["popularity"].isna()
nonnumeric = movies["popularity"].notna() & popularity_numeric.isna()
infinite = popularity_numeric.isin([np.inf, -np.inf])
negative = popularity_numeric.lt(0)
zero = popularity_numeric.eq(0)

checks = pd.Series({
    "Total rows": len(movies),
    "Missing popularity": int(missing.sum()),
    "Nonnumeric popularity": int(nonnumeric.sum()),
    "Infinite popularity": int(infinite.sum()),
    "Negative popularity": int(negative.sum()),
    "Zero popularity": int(zero.sum()),
})

display(checks.to_frame(name="count"))

# Exclude infinity from the descriptive statistics.
finite_popularity = popularity_numeric[
    popularity_numeric.notna() & ~infinite
]

print("Popularity summary:")
display(
    finite_popularity.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ).to_frame(name="popularity")
)

print("Ten highest popularity values:")
display(
    movies.assign(popularity_numeric=popularity_numeric)
    .nlargest(10, "popularity_numeric")
    [["id", "release_date", "popularity_numeric"]]
)

print("Examples of missing or invalid popularity:")
problem_rows = missing | nonnumeric | infinite | negative

display(
    movies.loc[
        problem_rows, ["id", "popularity"]
    ].head(10)
)

,count
Total rows,45449
Missing popularity,6
Nonnumeric popularity,0
Infinite popularity,0
Negative popularity,0
Zero popularity,66


Popularity summary:


,popularity
count,45443.000000
mean,2.921510
std,6.006106
min,0.000000
50%,1.127613
90%,8.328683
95%,11.061462
99%,17.006426
max,547.488298


Ten highest popularity values:


,id,release_date,popularity_numeric
30688,211672.0,2015-06-17,547.488298
33344,297762.0,2017-05-30,294.337037
42206,321612.0,2017-03-16,287.253654
43628,339403.0,2017-06-28,228.032744
24446,177572.0,2014-10-24,213.849907
26554,293660.0,2016-02-09,187.860492
26556,283995.0,2017-04-19,185.330992
14547,19995.0,2009-12-10,185.070892
24342,245891.0,2014-10-22,183.870374
23666,210577.0,2014-10-01,154.801009


Examples of missing or invalid popularity:


,id,popularity
19724,82663.0,NaN
19725,NaN,NaN
29490,122662.0,NaN
29491,NaN,NaN
35574,249260.0,NaN
35575,NaN,NaN


## Step 4: Create the Log-Popularity Target

**Current: Requirement 4 — Create `target_log_popularity` using `log1p(popularity)`.**

(Also addressing Requirement 3: Handle missing popularity values.)

(Also addressing Requirement 7: Document all feature sources and decisions.)

### Findings and Decisions

- Three movies have missing popularity; exclude them from the
  training-ready table because their target is unknown.
- Keep all 66 zero values because zero is valid.
- Keep unusually large values because magnitude alone does not
  establish an error.
- Preserve the original base table in `movies`.

Create a separate table named `features` and calculate:

`target_log_popularity = ln(1 + popularity)`

This transformation compresses large values while supporting zero.
It does not guarantee a normal distribution or remove outliers.


In [9]:
# Select movies with a known, finite, nonnegative target.
valid_target = (
    popularity_numeric.notna()
    & np.isfinite(popularity_numeric)
    & popularity_numeric.ge(0)
)

# Create a separate working table; preserve the original movies table.
features = movies.loc[valid_target].copy()
features["popularity"] = popularity_numeric.loc[valid_target]

# Remove duplicate movie IDs, keeping the first occurrence.
original_count = len(features)
features = features.drop_duplicates(subset=["id"], keep="first").copy()
duplicates_removed = original_count - len(features)

# Create the target required by Issue #13.
features["target_log_popularity"] = np.log1p(
    features["popularity"]
)

print("Original rows:", len(movies))
print("Rows excluded for invalid or missing targets:", int((~valid_target).sum()))
print("Duplicate movie IDs removed:", duplicates_removed)
print("Rows retained:", len(features))

print("\nOriginal popularity and transformed target:")
display(
    features[["id", "popularity", "target_log_popularity"]].head()
)

print("\nTarget summary:")
display(
    features["target_log_popularity"]
    .describe()
    .to_frame(name="target_log_popularity")
)

# Stop with an error if an important condition fails.
assert features["id"].notna().all(), "Missing movie IDs found."
assert features["id"].is_unique, "Duplicate movie IDs found."
assert np.isfinite(features["target_log_popularity"]).all(), (
    "Missing or infinite target values found."
)
assert features["target_log_popularity"].ge(0).all(), (
    "Negative target values found."
)

print("\nPassed: IDs are present and unique; targets are finite and nonnegative.")

Original rows: 45449
Rows excluded for invalid or missing targets: 6
Duplicate movie IDs removed: 13
Rows retained: 45430

Original popularity and transformed target:


,id,popularity,target_log_popularity
0,862.0,21.946943,3.133185
1,8844.0,17.015539,2.891235
2,15602.0,11.712900,2.542617
3,31357.0,3.859495,1.580935
4,11862.0,8.387519,2.239381



Target summary:


,target_log_popularity
count,45430.000000
mean,0.978699
std,0.803990
min,0.000000
25%,0.326330
50%,0.754824
75%,1.542898
max,6.307166



Passed: IDs are present and unique; targets are finite and nonnegative.


## Step 5: Inspect Additional Data Sources

**Current: Requirement 2 — Retrieve additional useful columns from `data/raw/` or `data/processed/` using `id`.**

(Also checking Requirement 6: Confirm one row per `id`.)

(Also addressing Requirement 7: Document all feature sources and decisions.)

Inspect the processed credits, nested metadata, movie text, and cleaned
keywords files before joining them to the base dataset.

Check each file's columns, missing IDs, duplicate IDs, and coverage of
the movies retained in `features`.

This step only inspects the sources. It does not join or change them.


In [10]:
# Load the additional processed datasets.
sources = {
    "credits": pd.read_csv(processed_folder / "credits.csv"),
    "nested_metadata": pd.read_csv(processed_folder / "nested_metadata.csv"),
    "movie_text": pd.read_csv(processed_folder / "movie_text.csv"),
    "keywords": pd.read_csv(processed_folder / "keywords_clean.csv"),
}

source_checks = []

for source_name, table in sources.items():
    print(f"\n{source_name} columns:")
    print(table.columns.tolist())

    # Inspect IDs using the same numeric representation as the base.
    source_ids = pd.to_numeric(table["id"], errors="coerce")

    valid_ids = (
        source_ids.notna()
        & np.isfinite(source_ids)
        & source_ids.mod(1).eq(0)
    )

    usable_ids = source_ids.loc[valid_ids].astype("int64")
    matched_movies = features["id"].isin(usable_ids)

    source_checks.append({
        "source": source_name,
        "rows": len(table),
        "missing_id_rows": int(table["id"].isna().sum()),
        "invalid_nonmissing_id_rows": int(
            (table["id"].notna() & ~valid_ids).sum()
        ),
        "duplicate_valid_id_rows": int(usable_ids.duplicated().sum()),
        "base_movies_matched": int(matched_movies.sum()),
        "base_movies_unmatched": int((~matched_movies).sum()),
    })

print("\nSource checks:")
display(pd.DataFrame(source_checks).set_index("source"))

print("\nExample values from each source:")
for source_name, table in sources.items():
    print(f"\n{source_name}:")
    display(table.head(2))


credits columns:
['id', 'director', 'top_five_cast', 'cast_size', 'crew_size']

nested_metadata columns:
['id', 'original_title', 'release_date', 'genres', 'belongs_to_collection', 'production_companies', 'production_countries', 'spoken_languages']

movie_text columns:
['id', 'title', 'original_title', 'overview', 'original_language', 'status']

keywords columns:
['id', 'keyword_names_clean']

Source checks:


,rows,missing_id_rows,invalid_nonmissing_id_rows,duplicate_valid_id_rows,base_movies_matched,base_movies_unmatched
source,,,,,,
credits,45432,0,0,0,45429,1
nested_metadata,45433,0,0,0,45430,0
movie_text,45433,0,0,0,45430,0
keywords,31092,0,0,0,31089,14341



Example values from each source:

credits:


,id,director,top_five_cast,cast_size,crew_size
0,862,John Lasseter,"[""Tom Hanks"", ""Tim Allen"", ""Don Rickles"", ""Jim...",13,106
1,8844,Joe Johnston,"[""Robin Williams"", ""Jonathan Hyde"", ""Kirsten D...",26,16



nested_metadata:


,id,original_title,release_date,genres,belongs_to_collection,production_companies,production_countries,spoken_languages
0,862,Toy Story,1995-10-30,Animation|Comedy|Family,Toy Story Collection,Pixar Animation Studios,United States of America,English
1,8844,Jumanji,1995-12-15,Adventure|Fantasy|Family,NaN,TriStar Pictures|Teitler Film|Interscope Commu...,United States of America,English|Français



movie_text:


,id,title,original_title,overview,original_language,status
0,2,Ariel,Ariel,Taisto Kasurinen is a Finnish coal miner whose...,fi,Released
1,3,Shadows in Paradise,Varjoja paratiisissa,"An episode in the life of Nikander, a garbage ...",fi,Released



keywords:


,id,keyword_names_clean
0,862,"['jealousy', 'toy', 'boy', 'friendship', 'frie..."
1,8844,"['board game', 'disappearance', ""based on chil..."


## Step 6: Join Additional Data by Movie ID

**Current: Requirement 2 — Retrieve additional useful columns using `id`.**

(Also checking Requirement 6: Confirm one row per `id`.)

(Also addressing Requirement 7: Document all feature sources and decisions.)

Use left joins to preserve all 45,430 movies with valid targets.

- Credits: one base movie has no matching record.
- Nested metadata and movie text: all base movies match.
- Keywords: 14,341 base movies have no matching cleaned record.

Preserve missing matches for later handling rather than treating them
as confirmed empty values.

Rename `top_five_cast` to `top_cast` and `keyword_names_clean` to
`keywords` to match the requested output names.

Keep the base release date and use original titles from movie_text
to avoid overlapping columns.


In [11]:
# Choose the columns we need from each source.
join_columns = {
    "credits": [
        "id", "director", "top_five_cast", "cast_size", "crew_size"
    ],
    "nested_metadata": [
        "id", "genres", "belongs_to_collection",
        "production_companies", "production_countries",
        "spoken_languages"
    ],
    "movie_text": [
        "id", "title", "original_title", "overview",
        "original_language", "status"
    ],
    "keywords": ["id", "keyword_names_clean"],
}

# Start from the base table with its validated target.
merged_features = features.copy()
join_results = []

for source_name, columns in join_columns.items():
    additional = sources[source_name][columns].copy()
    additional["id"] = pd.to_numeric(
        additional["id"], errors="raise"
    ).astype("int64")

    merged_features = merged_features.merge(
        additional,
        on="id",
        how="left",
        validate="one_to_one",
    )

    join_results.append({
        "source_added": source_name,
        "rows_after_join": len(merged_features),
    })

merged_features = merged_features.rename(columns={
    "top_five_cast": "top_cast",
    "keyword_names_clean": "keywords",
})

assert len(merged_features) == len(features)
assert merged_features["id"].notna().all()
assert merged_features["id"].is_unique

display(pd.DataFrame(join_results))
print("Total columns:", len(merged_features.columns))
print("Passed: all base movies retained, with one row per ID.")

,source_added,rows_after_join
0,credits,45430
1,nested_metadata,45430
2,movie_text,45430
3,keywords,45430


Total columns: 22
Passed: all base movies retained, with one row per ID.


## Step 7: Create Release Date Features

**Current: Requirement 5 — Create the required columns: `release_year`, `release_month`, and `release_decade`.**

(Also addressing Requirement 7: Document all feature sources and decisions.)

Source: `release_date` from processed movies_metadata.csv.

Convert release dates to datetime values, then extract the year,
month, and decade. Represent each decade by its starting year,
such as 1990 for the 1990s.

Keep missing or invalid dates as missing rather than inventing dates.
Count missing source dates separately from dates that fail parsing.


In [12]:
# Parse dates without changing the original release_date column.
release_dates = pd.to_datetime(
    merged_features["release_date"],
    format="%Y-%m-%d",
    errors="coerce",
)

# Nullable integers allow whole numbers and missing values.
merged_features["release_year"] = release_dates.dt.year.astype("Int64")
merged_features["release_month"] = release_dates.dt.month.astype("Int64")
merged_features["release_decade"] = (
    (merged_features["release_year"] // 10) * 10
).astype("Int64")

source_missing = merged_features["release_date"].isna()
parse_failed = (
    merged_features["release_date"].notna()
    & release_dates.isna()
)

print("Missing source dates:", int(source_missing.sum()))
print("Nonmissing dates that failed parsing:", int(parse_failed.sum()))

display(
    merged_features[
        ["id", "release_date", "release_year",
         "release_month", "release_decade"]
    ].head()
)

# Check every available month is between 1 and 12.
assert merged_features["release_month"].dropna().between(1, 12).all()

# Confirm that deriving features preserved the movie rows.
assert len(merged_features) == len(features)
assert merged_features["id"].is_unique

print("Passed: valid month range; row count and unique IDs preserved.")

Missing source dates: 84
Nonmissing dates that failed parsing: 0


,id,release_date,release_year,release_month,release_decade
0,862.0,1995-10-30,1995,10,1990
1,8844.0,1995-12-15,1995,12,1990
2,15602.0,1995-12-22,1995,12,1990
3,31357.0,1995-12-22,1995,12,1990
4,11862.0,1995-02-10,1995,2,1990


Passed: valid month range; row count and unique IDs preserved.


## Step 8: Inspect Financial and Runtime Values

**Current: Requirement 5 — Prepare `log_budget`, `log_revenue`, and `runtime`.**

(Also addressing Requirement 7: Document all feature sources and decisions.)

Source: `budget`, `revenue`, and `runtime` from processed movies_metadata.csv.

Inspect missing, nonnumeric, infinite, negative, and zero values
before creating financial features.

Zero budget or revenue may represent unreported information.
Zero runtime is suspicious because it does not describe a usable
movie duration. Inspect these cases before deciding how to handle them.

This step does not change the source columns.


In [13]:
numeric_columns = ["budget", "revenue", "runtime"]
numeric_values = {}
numeric_checks = []

for column in numeric_columns:
    original = merged_features[column]
    values = pd.to_numeric(original, errors="coerce")
    numeric_values[column] = values

    numeric_checks.append({
        "column": column,
        "missing": int(original.isna().sum()),
        "nonnumeric": int(
            (original.notna() & values.isna()).sum()
        ),
        "infinite": int(values.isin([np.inf, -np.inf]).sum()),
        "negative": int(values.lt(0).sum()),
        "zero": int(values.eq(0).sum()),
    })

display(pd.DataFrame(numeric_checks).set_index("column"))

# Summarize finite values only.
numeric_preview = pd.DataFrame(numeric_values).replace(
    [np.inf, -np.inf], np.nan
)

display(
    numeric_preview.describe(percentiles=[0.50, 0.99])
    .loc[["count", "min", "50%", "99%", "max"]]
)

,missing,nonnumeric,infinite,negative,zero
column,,,,,
budget,0,0,0,0,36550
revenue,0,0,0,0,38032
runtime,257,0,0,0,1558


,budget,revenue,runtime
count,45430.0,4.543000e+04,45173.0
min,0.0,0.000000e+00,0.0
50%,0.0,0.000000e+00,95.0
99%,87000000.0,2.732157e+08,185.0
max,380000000.0,2.787965e+09,1256.0


## Step 9: Create Financial and Runtime Features

**Current: Requirement 5 — Create `log_budget`, `log_revenue`, and `runtime`.**

(Also addressing Requirement 7: Document all feature sources and decisions.)

Source: budget, revenue, and runtime from processed movies_metadata.csv.

### Decisions

- Treat zero budget and revenue as unknown for financial features.
  This is an assumption because the source does not distinguish
  unreported amounts from genuine zeros.
- Apply log1p to positive, finite budget and revenue values.
- Add budget_reported and revenue_reported flags: 1 means a positive,
  finite amount is available; 0 means it is unavailable.
- Preserve original budget and revenue columns for inspection.
- Treat zero runtime as missing and retain existing missing runtimes.
- Retain large positive runtimes pending evidence that they are errors.
- Do not fill missing values with averages at this stage.

Revenue is included because the issue requests it. It would not be
available for a prediction made before release, so later modeling must
choose predictors according to the intended prediction time.


In [14]:
# Create log financial features from positive, finite amounts.
for column in ["budget", "revenue"]:
    values = numeric_values[column]

    reported = (
        values.notna()
        & np.isfinite(values)
        & values.gt(0)
    )

    merged_features[f"{column}_reported"] = reported.astype("Int64")
    merged_features[f"log_{column}"] = np.log1p(
        values.where(reported)
    )

# Keep positive, finite runtimes; represent other values as missing.
runtime_values = numeric_values["runtime"]

valid_runtime = (
    runtime_values.notna()
    & np.isfinite(runtime_values)
    & runtime_values.gt(0)
)

merged_features["runtime"] = runtime_values.where(valid_runtime)

# Show a compact summary.
financial_columns = ["log_budget", "log_revenue", "runtime"]

display(
    merged_features[financial_columns].agg(
        ["count", "min", "median", "max"]
    ).T
)

print("Missing values:")
display(
    merged_features[financial_columns]
    .isna()
    .sum()
    .to_frame(name="missing")
)

# Check available feature values and preserve one row per movie.
for column in financial_columns:
    available = merged_features[column].dropna()
    assert np.isfinite(available).all()
    assert available.gt(0).all()

assert len(merged_features) == len(features)
assert merged_features["id"].is_unique

print("Passed: available values are positive and finite; movie rows preserved.")

,count,min,median,max
log_budget,8880.0,0.693147,15.894952,19.755682
log_revenue,7398.0,0.693147,16.638925,21.748578
runtime,43615.0,1.000000,95.000000,1256.000000


Missing values:


,missing
log_budget,36550
log_revenue,38032
runtime,1815


Passed: available values are positive and finite; movie rows preserved.


## Step 10: Create Production Features

**Current: Requirement 5 — Create `genre_count`, `has_collection`, `company_count`, `country_count`, and `spoken_language_count`.**

(Also addressing Requirement 7: Document all feature sources and decisions.)

Source: processed nested_metadata.csv, joined using id.

Genres, production companies, production countries, and spoken
languages are stored as names separated by the pipe character (`|`).

Count nonempty names in each field. For this processed source, interpret
a blank field as no listed entries and assign a count of zero.
This means “none listed,” not proof that none exist.

Set has_collection to 1 when a nonblank collection name is listed,
and 0 otherwise.

Preserve the source text columns for later modeling and inspection.


In [15]:
def count_listed_names(value):
    """Count nonempty names in a pipe-separated source field."""
    if pd.isna(value):
        return 0

    names = [
        name.strip()
        for name in str(value).split("|")
        if name.strip()
    ]
    return len(names)


production_sources = {
    "genre_count": "genres",
    "company_count": "production_companies",
    "country_count": "production_countries",
    "spoken_language_count": "spoken_languages",
}

for feature_name, source_column in production_sources.items():
    merged_features[feature_name] = (
        merged_features[source_column]
        .apply(count_listed_names)
        .astype("Int64")
    )

collection_names = (
    merged_features["belongs_to_collection"]
    .fillna("")
    .str.strip()
)

merged_features["has_collection"] = (
    collection_names.ne("").astype("Int64")
)

production_columns = [
    *production_sources,
    "has_collection",
]

display(
    merged_features[production_columns].agg(
        ["min", "median", "max"]
    ).T
)

print(
    "Movies with a listed collection:",
    int(merged_features["has_collection"].sum()),
)

for column in production_sources:
    assert merged_features[column].ge(0).all()

assert merged_features["has_collection"].isin([0, 1]).all()
assert len(merged_features) == len(features)
assert merged_features["id"].is_unique

print("Passed: production features are valid; movie rows preserved.")

,min,median,max
genre_count,0.0,2.0,8.0
company_count,0.0,1.0,26.0
country_count,0.0,1.0,25.0
spoken_language_count,0.0,1.0,18.0
has_collection,0.0,0.0,1.0


Movies with a listed collection: 4487
Passed: production features are valid; movie rows preserved.


## Step 11: Prepare Credits Features

**Current: Requirement 5 — Prepare `cast_size`, `crew_size`, and `has_director`.**

(Also addressing Requirement 7: Document all feature sources and decisions.)

Source: processed credits.csv from Task 1, joined using id.

- Reuse cast_size and crew_size calculated during credits cleaning.
- Keep zero counts when an existing credits record lists no members.
- Set has_director to 1 when a nonblank director name is listed.
- Set has_director to 0 when a matching credits record has no director.
- Leave credits features missing for the movie without a matching record.
- Keep top_cast, renamed from the Task 1 column top_five_cast.

Missing credits must not be interpreted as a confirmed empty cast
or crew.


In [16]:
# Identify movies that have a record in the credits source.
credits_ids = pd.to_numeric(
    sources["credits"]["id"], errors="raise"
).astype("int64")

credits_matched = merged_features["id"].isin(credits_ids)

# Validate existing counts before converting to nullable integers.
for column in ["cast_size", "crew_size"]:
    values = pd.to_numeric(
        merged_features[column], errors="raise"
    )
    available = values.dropna()

    assert np.isfinite(available).all()
    assert available.ge(0).all()
    assert available.mod(1).eq(0).all()

    merged_features[column] = values.astype("Int64")

# Distinguish no listed director from an unmatched credits record.
director_names = (
    merged_features["director"]
    .fillna("")
    .str.strip()
)

merged_features["has_director"] = (
    director_names.ne("").astype("Int64")
).where(credits_matched, pd.NA)

credits_columns = ["cast_size", "crew_size", "has_director"]

display(
    merged_features[credits_columns].agg(
        ["count", "min", "median", "max"]
    ).T
)

print("Movies without a credits record:", int((~credits_matched).sum()))

print("\nhas_director counts (including missing):")
display(
    merged_features["has_director"]
    .value_counts(dropna=False)
    .to_frame(name="movies")
)

assert merged_features.loc[
    ~credits_matched, credits_columns
].isna().all().all()

assert merged_features["has_director"].dropna().isin([0, 1]).all()
assert len(merged_features) == len(features)
assert merged_features["id"].is_unique

print("Passed: credits features validated; unmatched records remain missing.")

,count,min,median,max
cast_size,45429.0,0.0,10.0,313.0
crew_size,45429.0,0.0,6.0,435.0
has_director,45429.0,0.0,1.0,1.0


Movies without a credits record: 1

has_director counts (including missing):


,movies
has_director,
1,44542
0,887
<NA>,1


Passed: credits features validated; unmatched records remain missing.


## Step 12: Create the Keyword Count

**Current: Requirement 5 — Create `keyword_count` and prepare the `keywords` modeling input.**

(Also addressing Requirement 7: Document all feature sources and decisions.)

Source: keyword_names_clean from processed keywords_clean.csv,
renamed to keywords during joining.

Parse the stored Python-style list text into actual lists and count
the keyword entries.

- Preserve missing keyword information as missing.
- Count an explicitly stored empty list as zero.
- Stop on malformed values rather than silently treating them as empty.
- Preserve the joined source text in merged_features["keywords"].

The 14,341 unmatched movies have no cleaned keyword record.
This does not establish that those movies have no keywords.


In [17]:
import ast

def parse_keywords(value):
    """Read a stored keyword list while preserving unknown values."""
    if pd.isna(value):
        return None

    parsed = ast.literal_eval(value)

    if not isinstance(parsed, list):
        raise ValueError("Expected a keyword list.")

    if not all(isinstance(word, str) for word in parsed):
        raise ValueError("Expected keyword strings.")

    # Four source lists contain a blank entry. Count only real names.
    # Preserve the original keywords column for inspection.
    return [word.strip() for word in parsed if word.strip()]


# Parse into a separate Series, preserving the source text column.
keyword_lists = merged_features["keywords"].apply(parse_keywords)

merged_features["keyword_count"] = keyword_lists.apply(
    lambda words: pd.NA if words is None else len(words)
).astype("Int64")

print("Movies with known keyword counts:",
      int(merged_features["keyword_count"].notna().sum()))
print("Movies with unknown keyword counts:",
      int(merged_features["keyword_count"].isna().sum()))
print("Movies with no nonblank keywords in a known list:",
      int(merged_features["keyword_count"].eq(0).sum()))

display(
    merged_features["keyword_count"]
    .describe()
    .to_frame(name="keyword_count")
)

assert merged_features["keyword_count"].dropna().ge(0).all()
assert (
    merged_features["keyword_count"].isna()
    == merged_features["keywords"].isna()
).all()

assert len(merged_features) == len(features)
assert merged_features["id"].is_unique

print("Passed: keyword lists parsed; missing information preserved.")

Movies with known keyword counts: 31089
Movies with unknown keyword counts: 14341
Movies with no nonblank keywords in a known list: 0


,keyword_count
count,31089.0
mean,5.036572
std,4.912839
min,1.0
25%,2.0
50%,4.0
75%,6.0
max,149.0


Passed: keyword lists parsed; missing information preserved.


## Step 13: Create Text Features

**Current: Requirement 5 — Create `title_word_count`, `overview_word_count`, and `title_differs_from_original`.**

(Also addressing Requirement 7: Document all feature sources and decisions.)

Source: title, original_title, and overview from processed movie_text.csv.

### Decisions

- Count whitespace-separated words in titles and overviews.
- Preserve missing text as missing counts.
- An available but empty string has a word count of zero.
- Compare titles after normalizing capitalization and whitespace.
- Set title_differs_from_original to 1 when normalized titles differ,
  and 0 when they match.
- Leave the comparison missing when either title is missing or blank.
- Preserve the original text columns.


In [18]:
# Count words without replacing missing text with empty strings.
for source_column, feature_name in {
    "title": "title_word_count",
    "overview": "overview_word_count",
}.items():
    text = merged_features[source_column].astype("string")

    merged_features[feature_name] = (
        text.str.split().str.len().astype("Int64")
    )

# Normalize titles for comparison without changing the source columns.
title_normalized = (
    merged_features["title"]
    .astype("string")
    .str.casefold()
    .str.split()
    .str.join(" ")
)

original_normalized = (
    merged_features["original_title"]
    .astype("string")
    .str.casefold()
    .str.split()
    .str.join(" ")
)

titles_available = (
    title_normalized.notna()
    & original_normalized.notna()
    & title_normalized.ne("")
    & original_normalized.ne("")
)

merged_features["title_differs_from_original"] = (
    title_normalized.ne(original_normalized)
    .astype("Int64")
    .where(titles_available, pd.NA)
)

text_features = [
    "title_word_count",
    "overview_word_count",
    "title_differs_from_original",
]

display(
    merged_features[text_features].agg(
        ["count", "min", "median", "max"]
    ).T
)

print("\nMissing text features:")
display(
    merged_features[text_features]
    .isna()
    .sum()
    .to_frame(name="missing")
)

for column in ["title_word_count", "overview_word_count"]:
    assert merged_features[column].dropna().ge(0).all()

assert (
    merged_features["title_differs_from_original"]
    .dropna()
    .isin([0, 1])
    .all()
)

assert len(merged_features) == len(features)
assert merged_features["id"].is_unique

print("Passed: text features validated; movie rows preserved.")

,count,min,median,max
title_word_count,45430.0,1.0,3.0,20.0
overview_word_count,44326.0,1.0,48.0,187.0
title_differs_from_original,45430.0,0.0,0.0,1.0



Missing text features:


,missing
title_word_count,0
overview_word_count,1104
title_differs_from_original,0


Passed: text features validated; movie rows preserved.


## Step 14: Validate the Final Feature Dataset

**Current: Requirement 6 — Confirm one row per `id`.**

(Also checking Requirement 5: Confirm all required columns exist.)

(Also addressing Requirement 7: Document validation results.)

Check that the dataset:

- Includes every required feature and modeling-input column.
- Preserves all movies with valid popularity targets.
- Has present, unique integer movie IDs.
- Has finite, nonnegative targets calculated as log1p(popularity).
- Contains no infinite numeric feature values.

Keep documented missing predictor values as missing.
Exclude raw popularity from the exported dataset because it directly
reveals the target. Retain log_revenue as requested, with its
prediction-time limitation documented in the report.


In [22]:
required_columns = [
    "id", "target_log_popularity",
    "release_year", "release_month", "release_decade",
    "log_budget", "log_revenue", "runtime",
    "genre_count", "has_collection", "company_count",
    "country_count", "spoken_language_count",
    "cast_size", "crew_size", "has_director",
    "keyword_count", "title_word_count", "overview_word_count",
    "title_differs_from_original",
    "genres", "production_companies", "production_countries",
    "spoken_languages", "director", "top_cast", "keywords",
    "title", "original_title", "overview",
    "original_language", "status",
]

missing_columns = [
    column for column in required_columns
    if column not in merged_features.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

# Select the required columns plus our justified reporting flags.
final_features = merged_features[
    required_columns + ["budget_reported", "revenue_reported"]
].copy()

# Convert movie IDs to integers.
ids = pd.to_numeric(final_features["id"], errors="raise")

assert ids.notna().all(), "Missing movie IDs found."
assert np.isfinite(ids).all(), "Infinite movie IDs found."
assert (ids % 1 == 0).all(), "Non-integer movie IDs found."

final_features["id"] = ids.astype("int64")

# Validate required columns and movie IDs.
assert final_features.columns.is_unique
assert final_features["id"].notna().all()
assert final_features["id"].is_unique
assert pd.api.types.is_integer_dtype(final_features["id"])
assert len(final_features) == len(features)

# Compare movie IDs after converting both to integers.
assert set(final_features["id"]) == set(
    pd.to_numeric(features["id"], errors="raise").astype("int64")
)

# Validate target values.
target = final_features["target_log_popularity"]

assert np.isfinite(target).all()
assert target.ge(0).all()

assert np.allclose(
    target.to_numpy(dtype=float),
    np.log1p(merged_features["popularity"].to_numpy(dtype=float)),
)

# Validate numeric columns.
numeric_output = final_features.select_dtypes(include="number")

assert not numeric_output.isin([np.inf, -np.inf]).any().any()
assert "popularity" not in final_features.columns

print("Final rows:", len(final_features))
print("Final columns:", len(final_features.columns))
print("Movie ID dtype:", final_features["id"].dtype)
print("Missing required columns:", missing_columns)
print("Passed: required columns, movie IDs, targets, and numeric checks.")

# Summarize missing values.
missing_counts = final_features.isna().sum()

print("\nColumns containing missing values:")
display(
    missing_counts[missing_counts.gt(0)]
    .sort_values(ascending=False)
    .to_frame(name="missing")
)

Final rows: 45430
Final columns: 34
Movie ID dtype: int64
Missing required columns: []
Passed: required columns, movie IDs, targets, and numeric checks.

Columns containing missing values:


,missing
log_revenue,38032
log_budget,36550
keywords,14341
keyword_count,14341
production_companies,11868
production_countries,6279
spoken_languages,3952
genres,2442
runtime,1815
overview_word_count,1104


## Step 15: Export and Verify the Feature Dataset

**Current: Deliverable — Save `data/features/09_popularity_features.csv`.**

(Also checking Requirement 6: Confirm one row per `id` after saving.)

(Also addressing Requirement 7: Document the output and validation.)

Save the validated feature table without a pandas row index.
Read the CSV back and verify its dimensions, column order, movie IDs,
missing-value counts, and numeric values.

CSV does not preserve pandas data types exactly, so validate values
while allowing inferred data types to differ.

The output stays local because the data folder is gitignored.


In [23]:
# Create the output folder if it does not already exist.
output_folder = project_root / "data" / "features"
output_folder.mkdir(parents=True, exist_ok=True)

output_path = output_folder / "09_popularity_features.csv"

# Save without adding an extra row-index column.
final_features.to_csv(output_path, index=False)

# Restore column types when reading: CSV does not store nullable types.
saved_features = pd.read_csv(
    output_path, dtype=final_features.dtypes.to_dict()
)

assert saved_features.shape == final_features.shape
assert saved_features.columns.tolist() == final_features.columns.tolist()
assert saved_features["id"].notna().all()
assert saved_features["id"].is_unique

# Compare restored column types and values, allowing tiny float differences.
pd.testing.assert_frame_equal(
    final_features.reset_index(drop=True),
    saved_features,
    check_dtype=True,
    check_exact=False,
    rtol=1e-10,
    atol=1e-12,
)

assert (
    saved_features.isna().sum()
    == final_features.isna().sum()
).all()

print("Saved file:", output_path)
print("Verified rows:", len(saved_features))
print("Verified columns:", len(saved_features.columns))
print("File size (MB):", round(output_path.stat().st_size / 1_000_000, 2))
print("Passed: saved CSV matches the validated feature table.")

Saved file: /Users/jasper/css-dsml-fall2027/data/features/09_popularity_features.csv
Verified rows: 45430
Verified columns: 34
File size (MB): 30.75
Passed: saved CSV matches the validated feature table.
